# soamp — Colab smoke test: build every representation, then overfit a tiny subset

Gate for `02_run_experiments.ipynb`. Two questions, answered by execution:

1. **Does every cell of the 2×2 grid actually construct?** All four
   (peptide × organism) featurizations are built on a small row sample, fed
   through `attention_fusion_classifier`, and checked for the dimensions and
   `output_kind` each is supposed to produce.
2. **Can each one drive the loss to zero?** Each cell trains on ~256 rows and
   evaluates on the same rows. A model that cannot memorise its own training
   set has a wiring bug — no point spending the real runs on it.

wandb is deliberately disabled here: a sanity check isn't an experiment.

## Before you run

A **GPU runtime** (Runtime → Change runtime type → T4 GPU). Not strictly
required — the classifier is tiny — but PeptideCLM featurization is ~15 min
on CPU versus seconds on a GPU.

Set these under **Colab Secrets** (the key icon in the left sidebar), with
notebook access enabled for each:

| Secret | Needed for |
|---|---|
| `GITHUB_TOKEN` | cloning the private repo (a fine-grained, read-only PAT is enough) |
| `WANDB_API_KEY` | logging runs to Weights & Biases |
| `WANDB_ENTITY` | optional; your wandb team/username if it isn't your default |

## 1. Environment

In [ ]:
import subprocess
import sys

print("Python:", sys.version.split()[0])
try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("GPU:", gpu)
except (FileNotFoundError, subprocess.CalledProcessError):
    print("GPU: none detected -- set Runtime > Change runtime type > T4 GPU, then rerun.")

In [ ]:
import os

from google.colab import userdata

# Colab Secrets (key icon in the left sidebar), not a committed .env: the repo
# gitignores .env, so nothing sensitive travels with the clone.
# WANDB_API_KEY is optional here -- this notebook never initialises a tracker.
REQUIRED = {"GITHUB_TOKEN": True, "WANDB_API_KEY": False, "WANDB_ENTITY": False}

for name, required in REQUIRED.items():
    try:
        os.environ[name] = userdata.get(name)
        print(f"{name}: set")
    except Exception as e:
        if required:
            raise RuntimeError(
                f"Colab secret {name!r} is missing. Add it under the key icon "
                f"in the left sidebar and enable notebook access."
            ) from e
        print(f"{name}: not set (optional)")

In [ ]:
import os
import subprocess

REPO = "LukaJinc/soamp"
BRANCH = "main"
WORKDIR = "/content/soamp"

if not os.path.isdir(WORKDIR):
    token = os.environ["GITHUB_TOKEN"]
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH,
         f"https://{token}@github.com/{REPO}.git", WORKDIR],
        capture_output=True, text=True,
    )
    # Scrub the token before anything reaches notebook output, which is saved
    # with the file.
    print((result.stdout + result.stderr).replace(token, "***"))
    if result.returncode != 0:
        raise RuntimeError("git clone failed -- check GITHUB_TOKEN has read access to the repo")
    # Drop the credential from .git/config too, so later git calls can't leak it.
    subprocess.run(
        ["git", "-C", WORKDIR, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"],
        check=True,
    )

os.chdir(WORKDIR)
print("HEAD:", subprocess.run(["git", "log", "-1", "--oneline"],
                              capture_output=True, text=True).stdout.strip())

In [ ]:
# requirements-colab.txt first, then the package with --no-deps: installing
# soamp's own pinned dependency set would replace Colab's CUDA-matched torch
# build and drag in curation-only packages this workload never imports.
!pip install -q -r scripts/colab/requirements-colab.txt
!pip install -q -e . --no-deps

In [ ]:
import torch

from soamp.data.factory import build_dataset
from soamp.model.factory import build_model
from soamp.utils.device import resolve_device

device = resolve_device("auto")
print(f"torch {torch.__version__} | cuda available: {torch.cuda.is_available()} | device: {device}")
if device.type != "cuda":
    print("\nWARNING: running on CPU. The classifier is small enough not to care, but the "
          "PeptideCLM featurization pass will take ~15min instead of seconds.")

## 2. Reference genomes

The `kmer_composition` organism featurization reads each organism's RefSeq
assembly. `.cache/` is gitignored, so the FASTAs don't come with the clone —
this fetches them (~13 MB, resumable, skips anything already cached).

Only this notebook needs them. `02_run_experiments.ipynb` reads the *committed*
`data/organism_vocab_kmer_composition.json`, which has the k-mer vectors baked
in, so the real runs need neither the genomes nor network access.

In [ ]:
!python pipeline/features/00_fetch_organism_genomes.py

## 3. Does every cell of the grid construct?

`build_dataset(row_groups=...)` computes featurization fresh from the rows
handed to it, fitting the scaler and organism vocab on the `fit` group only.
That's the same function `pipeline/train.py` calls (in its other mode, off the
precomputed artifacts), so anything that works here works there.

Expected per cell:

| method | gives |
|---|---|
| `rdkit_descriptors` | 13-dim peptide vector |
| `peptideclm_embedding` | 768-dim peptide vector |
| `vocab_embedding` | `output_kind="index"` → `nn.Embedding` inside the model |
| `kmer_composition` | `output_kind="vector"`, 340 dims → `nn.Linear` instead |

In [ ]:
import itertools

import pandas as pd
from torch.utils.data import DataLoader

ARCHITECTURE = "attention_fusion_classifier"
PEPTIDE_METHODS = ["rdkit_descriptors", "peptideclm_embedding"]
ORGANISM_METHODS = ["vocab_embedding", "kmer_composition"]
PROBE_N = 300

EXPECTED_PEPTIDE_DIM = {"rdkit_descriptors": 13, "peptideclm_embedding": 768}
EXPECTED_ORGANISM_KIND = {"vocab_embedding": "index", "kmer_composition": "vector"}

df = pd.read_csv("data/mic_classification_dataset.csv")
print(f"{len(df)} rows, {df['peptide_id'].nunique()} unique peptides, "
      f"organisms: {sorted(df['organism'].unique())}")


def peptide_kwargs(method):
    """PeptideCLM is the only featurizer with anything to configure per-run."""
    return {"device": str(device), "batch_size": 64} if method == "peptideclm_embedding" else None

In [ ]:
probe_rows = df.sample(n=PROBE_N, random_state=0).to_dict("records")

construction = []
for peptide_method, organism_method in itertools.product(PEPTIDE_METHODS, ORGANISM_METHODS):
    bundle = build_dataset(
        row_groups={"fit": probe_rows},
        peptide_method=peptide_method,
        peptide_method_kwargs=peptide_kwargs(peptide_method),
        organism_method=organism_method,
    )
    featurization = bundle.featurization
    model = build_model(bundle, architecture=ARCHITECTURE)

    peptide_features, organism_input, labels = next(
        iter(DataLoader(bundle.datasets["fit"], batch_size=8))
    )
    logits = model(peptide_features, organism_input)

    assert logits.shape == (8,), f"expected (8,) logits, got {tuple(logits.shape)}"
    assert featurization.peptide_feature_dim == EXPECTED_PEPTIDE_DIM[peptide_method]
    assert featurization.organism_output_kind == EXPECTED_ORGANISM_KIND[organism_method]

    construction.append({
        "peptide_method": peptide_method,
        "organism_method": organism_method,
        "peptide_dim": featurization.peptide_feature_dim,
        "organism_kind": featurization.organism_output_kind,
        "organism_dim": featurization.organism_feature_dim,
        "organism_vocab_size": featurization.organism_vocab_size,
        "params": sum(p.numel() for p in model.parameters()),
    })

print("all 4 cells construct and forward cleanly\n")
pd.DataFrame(construction)

## 4. Can each cell overfit 256 rows?

Train and evaluate on the **same** rows. This is a memorisation check, so
class balancing is off — `pos_weight` would rescale the loss we assert on.

A cell that plateaus well above zero here is telling you its representation
can't separate its own training rows: either the features are degenerate (an
all-zero organism vector from a failed genome lookup, say) or something
upstream is mismatched. Either way, stop and fix it before notebook 02.

In [ ]:
import torch

from soamp.engine.metrics import compute_binary_metrics
from soamp.engine.trainer import Trainer
from soamp.utils.reproducibility import seed_everything

OVERFIT_N = 256
OVERFIT_EPOCHS = 300
OVERFIT_LR = 3e-3
LOSS_TARGET = 0.01
SEED = 42


def overfit_one(peptide_method, organism_method):
    """Returns (per-epoch loss curve, metrics on the fit rows themselves)."""
    seed_everything(SEED)
    rows = df.sample(n=OVERFIT_N, random_state=SEED).to_dict("records")
    label_counts = pd.Series([r["label"] for r in rows]).value_counts()
    assert len(label_counts) == 2, f"need both classes to score AUROC, got {dict(label_counts)}"

    bundle = build_dataset(
        row_groups={"fit": rows},
        peptide_method=peptide_method,
        peptide_method_kwargs=peptide_kwargs(peptide_method),
        organism_method=organism_method,
    )
    model = build_model(bundle, architecture=ARCHITECTURE)

    # Full-batch, shuffled: small enough that a single batch per epoch is fine.
    loader = DataLoader(bundle.datasets["fit"], batch_size=OVERFIT_N, shuffle=True)
    trainer = Trainer(
        model,
        torch.optim.Adam(model.parameters(), lr=OVERFIT_LR),
        torch.nn.BCEWithLogitsLoss(),  # no pos_weight -- see markdown above
        device=device,
    )

    curve = [trainer.train_epoch(loader)["loss"] for _ in range(OVERFIT_EPOCHS)]
    evaluated = trainer.evaluate(DataLoader(bundle.datasets["fit"], batch_size=OVERFIT_N))
    return curve, compute_binary_metrics(evaluated["logits"], evaluated["labels"])

In [ ]:
curves, overfit_rows = {}, []
for peptide_method, organism_method in itertools.product(PEPTIDE_METHODS, ORGANISM_METHODS):
    cell = f"{peptide_method} x {organism_method}"
    curve, metrics = overfit_one(peptide_method, organism_method)
    curves[cell] = curve
    overfit_rows.append({
        "cell": cell,
        "first_loss": curve[0],
        "final_loss": curve[-1],
        "fit_accuracy": metrics["accuracy"],
        "fit_auroc": metrics["auroc"],
        "reached_zero": curve[-1] < LOSS_TARGET,
    })
    print(f"{cell:48s} loss {curve[0]:.4f} -> {curve[-1]:.5f}  "
          f"acc {metrics['accuracy']:.3f}")

overfit_df = pd.DataFrame(overfit_rows)
overfit_df

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))
for cell, curve in curves.items():
    ax.plot(curve, label=cell, linewidth=1.6)
ax.axhline(LOSS_TARGET, color="0.4", linestyle="--", linewidth=1,
           label=f"target ({LOSS_TARGET})")
ax.set_yscale("log")
ax.set_xlabel("epoch")
ax.set_ylabel("train loss (log scale)")
ax.set_title(f"Overfitting {OVERFIT_N} rows, {ARCHITECTURE}")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
failed = overfit_df[~overfit_df["reached_zero"]]
if len(failed):
    raise AssertionError(
        f"{len(failed)} cell(s) did not reach loss < {LOSS_TARGET}:\n{failed.to_string(index=False)}\n\n"
        "Raise OVERFIT_EPOCHS/OVERFIT_LR first -- if it still plateaus, that "
        "representation has a wiring problem worth fixing before the real runs."
    )
print(f"All {len(overfit_df)} cells drove the loss below {LOSS_TARGET}.")
print("Gate passed -- 02_run_experiments.ipynb is safe to run.")

## What this did and didn't prove

**Did:** every featurization builds, produces the dimensions it claims, flows
through `attention_fusion_classifier`, and has enough capacity to memorise its
own training rows on this runtime.

**Didn't:** say anything about generalisation. Fitting 256 rows perfectly is the
*absence of a bug*, not evidence of a good representation — that question is
`02_run_experiments.ipynb`'s, on held-out test rows the model never sees during
training.